# 07b · Gün İçi Stratejiler: ORB ve Alternatifleri
## Bölüm 2 — Sabit evren, 5 dk mı 15 dk mı, hangi teyit işe yarıyor?

Bölüm 1'in (07) devamıdır. Bölüm 1 bulguları: 15 dk ORB makale ayarıyla (ATR %10 stop) 5 dk veride **belirsiz** çıktı; 5 dk aralık her ayarda daha iyiydi; **göreli hacmi yüksek** günler belirgin biçimde ayrıştı; tekil hisselerde (NVDA, TSLA) dar stop gürültüde boğuldu.

**Bu bölümün kuralları (KARARLAR.md, 07):**
- **Evren sabit:** SPY, QQQ, IWM ve Mag7 (AAPL, MSFT, NVDA, AMZN, GOOGL, META, TSLA). Liste genişletilmez; küçük ve manipülasyona açık hisseler yanıltıcı sonuç verir.
- Sonuçlar **hisse hisse** raporlanır; ETF'ler ve Mag7 ayrı gruptur. Bir sembolde işleyen kural ötekine taşınmış sayılmaz.
- 5 dk ve 15 dk aralık **yan yana** test edilir.

**Test edilen teyitler (her biri tek başına):**

| Teyit | Fikir | Kaynak |
|---|---|---|
| Göreli hacim | Açılış hacmi alışılmadık yüksekse kırılım daha sağlıklı | Zarattini ve ark. 2024 ("Stocks in Play") |
| VIX medyan üstü | Gün içi momentum volatil günlerde daha güçlü | Gao ve ark. 2018 (JFE) |
| Gece aralığı dışına kırılım | Vadeli gece yükseği/düşüğü aşıldıysa kırılım "açık havada" | Uygulayıcı görüşü, sayısal testi yok |
| Açılış gece orta noktasının işlem yönünde | Açılış gece aralığının üst yarısındaysa yukarı kırılım daha olası | NQ istatistiği (uygulayıcı) |
| Gece getirisi işlem yönünde | Vadelinin önceki kapanıştan açılışa hareketi ORB yönüyle aynı | Gao ve ark.: ilk yarım saat getirisi önceki kapanıştan ölçülür |
| ES ve NQ aralığı aynı yönde | İki endeks açılışta aynı yönde anlaşıyor (SMT'nin tersi) | Kamuya açık testi yok |
| VWAP iz süren çıkış | VWAP girişte değil, çıkışta kullanılır | Noise Area makalesi |

**Neden VWAP giriş filtresi değil?** 15 dk sonunda VWAP ilk 15 dakikanın hacim ağırlıklı ortalamasıdır; matematiksel olarak aralığın içindedir. Aralık yükseğini kıran fiyat zaten VWAP'ın üstündedir, filtre hiçbir şeyi elemez.

**Vadeli eşleştirme:** SPY → ES, QQQ → NQ, IWM → RTY, Mag7 → NQ. Vadeliler sinyal değil, **gün bağlamı** olarak kullanılır (gece seansı ETF kapalıyken oluşur).

**Kullanım:** Ayarları gözden geçir, **Çalışma zamanı > Tümünü çalıştır** (Ctrl+F9). API anahtarı gerekmez. Rastgele kıyaslar nedeniyle birkaç dakika sürebilir.

> ⚠️ Öğrenme amaçlıdır, yatırım tavsiyesi değildir.

In [ ]:
#@title ⚙️ Ayarlar
ETF_LER = "SPY, QQQ, IWM" #@param {type:"string"}
MAG7 = "AAPL, MSFT, NVDA, AMZN, GOOGL, META, TSLA" #@param {type:"string"}
#@markdown Evren KARARLAR.md'de sabittir; değiştirmeden önce kararı güncelle.
STOP_TURU = "aralığın karşı ucu" #@param ["aralığın karşı ucu", "aralığın ortası", "ATR yüzdesi (makale)"]
#@markdown Varsayılan geniş stop: 5 dk veride mum içi belirsizliği küçültür (Bölüm 1).
ATR_YUZDE = 10 #@param {type:"number"}
YON_KURALI = "aralık yönü (makale)" #@param ["aralık yönü (makale)", "ilk kırılan taraf"]
SON_GIRIS = "15:30" #@param {type:"string"}
#@markdown ---
#@markdown **Filtre eşikleri**
GH_ESIK = 1.5 #@param {type:"number"}
#@markdown Göreli hacim eşiği (açılış aralığı hacmi ÷ önceki 14 günün ortalaması).
#@markdown ---
#@markdown **Maliyet** (her yön)
KOMISYON_HISSE = 0.0035 #@param {type:"number"}
KAYMA_BPS_ETF = 1 #@param {type:"number"}
KAYMA_BPS_HISSE = 3 #@param {type:"number"}
#@markdown ---
RASTGELE_TEKRAR = 300 #@param {type:"integer"}
PERMUTASYON_TEKRAR = 2000 #@param {type:"integer"}
TOHUM = 42
VADELI_ESLEME = {"SPY": "ES=F", "QQQ": "NQ=F", "IWM": "RTY=F"}   # Mag7 -> NQ=F

## 1 · Kurulum ve yardımcılar

In [ ]:
import subprocess, sys
try:
    import yfinance as yf
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance"], check=True)
    import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def tr_sayi(x, ond=2):
    if x is None or (isinstance(x, (float, np.floating)) and np.isnan(x)):
        return "–"
    s = f"{abs(x):,.{ond}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return ("-" if x < 0 else "") + s

def tr_yuzde(x, ond=1):
    if x is None or (isinstance(x, (float, np.floating)) and np.isnan(x)):
        return "–"
    return ("-" if x < 0 else "") + "%" + tr_sayi(abs(x) * 100, ond)

def tr_tarih(d):
    return pd.Timestamp(d).strftime("%d/%m/%Y")

def dk(saat):
    s, d = saat.split(":")
    return int(s) * 60 + int(d)

def tr_tablo(t, tam=("işlem", "n", "geçti n", "kaldı n"), yuzde=("isabet",), ond=3):
    b = t.copy().astype(object)
    for col in t.columns:
        for idx in t.index:
            x = t.at[idx, col]
            if isinstance(x, str):
                continue
            if col in tam:
                b.at[idx, col] = tr_sayi(x, 0)
            elif any(col.startswith(y) for y in yuzde):
                b.at[idx, col] = tr_yuzde(x)
            else:
                b.at[idx, col] = tr_sayi(x, ond)
    return b

ACILIS_DK = dk("09:30")
etfler = [s.strip().upper() for s in ETF_LER.split(",") if s.strip()]
mag7 = [s.strip().upper() for s in MAG7.split(",") if s.strip()]
semboller = etfler + mag7
GRUP = {s: ("ETF" if s in etfler else "Mag7") for s in semboller}
VADELI = {s: VADELI_ESLEME.get(s, "NQ=F") for s in semboller}
rng = np.random.default_rng(TOHUM)
print("Evren:", ", ".join(semboller))

## 2 · Veri
- **ETF / hisse:** 5 dk mumlar (son ~60 gün, yalnızca seans), günlük mumlar (ATR ve önceki kapanış).
- **VIX:** günlük kapanış; her gün için **bir önceki** kapanış kullanılır.
- **Vadeliler (ES, NQ, RTY):** 5 dk mumlar, **gece seansı dahil**. İlk iş, gece verisinin gerçekten geldiğini doğrulamak.
- **Bilanço tarihleri (Mag7):** kapanış sonrası açıklanan bilanço ertesi günün, açılış öncesi açıklanan aynı günün "bilanço günü" sayılır.

In [ ]:
def seans_mumlari(sembol):
    t = yf.Ticker(sembol)
    m5 = t.history(period="59d", interval="5m", prepost=False, auto_adjust=False)
    gun = t.history(period="1y", interval="1d", auto_adjust=False)
    if m5.empty or gun.empty:
        raise ValueError("veri gelmedi")
    m5.index = m5.index.tz_convert("America/New_York")
    m5 = m5[["Open", "High", "Low", "Close", "Volume"]]
    dakika = m5.index.hour * 60 + m5.index.minute
    m5 = m5[(dakika >= ACILIS_DK) & (dakika < dk("16:00"))]
    gun.index = pd.to_datetime(gun.index.tz_convert("America/New_York").date)
    onceki = gun["Close"].shift(1)
    tr_ = pd.concat([gun["High"] - gun["Low"], (gun["High"] - onceki).abs(),
                     (gun["Low"] - onceki).abs()], axis=1).max(axis=1)
    gun["ATR14"] = tr_.rolling(14).mean().shift(1)
    gun["OncekiKapanis"] = onceki
    return m5, gun

def gunlere_bol(m5, gun):
    gunler = []
    for tarih, g in m5.groupby(m5.index.date):
        tarih = pd.Timestamp(tarih)
        if tarih not in gun.index or np.isnan(gun.at[tarih, "ATR14"]):
            continue
        dakika = (g.index.hour * 60 + g.index.minute).values
        if dakika[0] != ACILIS_DK:
            continue
        tp = (g["High"].values + g["Low"].values + g["Close"].values) / 3
        v = g["Volume"].values.astype(float)
        kum_v = np.cumsum(v)
        vwap = np.where(kum_v > 0, np.cumsum(tp * v) / np.where(kum_v > 0, kum_v, 1), tp)
        gunler.append(dict(tarih=tarih, dk=dakika, o=g["Open"].values, h=g["High"].values,
                           l=g["Low"].values, c=g["Close"].values, v=v, vwap=vwap,
                           atr=gun.at[tarih, "ATR14"], onceki_kapanis=gun.at[tarih, "OncekiKapanis"]))
    return gunler

VERI = {}
for s in semboller:
    try:
        m5, gun = seans_mumlari(s)
        VERI[s] = gunlere_bol(m5, gun)
        print(f"{s:5s}: {len(VERI[s])} gün, {tr_tarih(VERI[s][0]['tarih'])} – {tr_tarih(VERI[s][-1]['tarih'])}")
    except Exception as e:
        print(f"{s:5s}: atlandı ({e})")

# VIX: bir önceki günün kapanışı
try:
    vix = yf.Ticker("^VIX").history(period="1y", interval="1d")["Close"]
    vix.index = pd.to_datetime(vix.index.tz_convert("America/New_York").date)
    VIX_ONCEKI = vix.shift(1)
    print(f"VIX: {len(vix)} gün, son kapanış {tr_sayi(vix.iloc[-1], 2)}")
except Exception as e:
    VIX_ONCEKI = pd.Series(dtype=float)
    print("VIX alınamadı:", e)

In [ ]:
def vadeli_gunleri(sembol):
    """Her işlem günü için gece aralığı (önceki gün 18:00 – 09:30 ET) ve seans mumları."""
    m = yf.Ticker(sembol).history(period="59d", interval="5m", prepost=True, auto_adjust=False)
    if m.empty:
        raise ValueError("veri gelmedi")
    m.index = m.index.tz_convert("America/New_York")
    dakika = m.index.hour * 60 + m.index.minute
    sonuc, gece_mum_sayisi = {}, []
    # Yalnızca normal seansı olan günler (Pazar akşamı açılışı ayrı bir gün sayılmaz);
    # Pazartesi'nin gece aralığı Cuma 18:00'den başlar.
    seansli = (dakika >= ACILIS_DK) & (dakika < dk("16:00"))
    gunler = sorted(set(m.index[seansli].date))
    for i in range(1, len(gunler)):
        bugun, dun = gunler[i], gunler[i - 1]
        bas = pd.Timestamp(dun).tz_localize("America/New_York") + pd.Timedelta(hours=18)
        acilis = pd.Timestamp(bugun).tz_localize("America/New_York") + pd.Timedelta(hours=9, minutes=30)
        kapanis = pd.Timestamp(bugun).tz_localize("America/New_York") + pd.Timedelta(hours=16)
        gece = m[(m.index >= bas) & (m.index < acilis)]
        seans = m[(m.index >= acilis) & (m.index < kapanis)]
        dun_seans = m[(m.index.date == dun) & (dakika >= ACILIS_DK) & (dakika < dk("16:00"))]
        if len(gece) < 12 or len(seans) < 12 or dun_seans.empty:
            continue
        gece_mum_sayisi.append(len(gece))
        sonuc[pd.Timestamp(bugun)] = dict(
            onh=gece["High"].max(), onl=gece["Low"].min(), p0930=gece["Close"].iloc[-1],
            onceki_kapanis=dun_seans["Close"].iloc[-1],
            dk=(seans.index.hour * 60 + seans.index.minute).values,
            o=seans["Open"].values, h=seans["High"].values, l=seans["Low"].values, c=seans["Close"].values)
    return sonuc, (np.mean(gece_mum_sayisi) if gece_mum_sayisi else 0)

VADELI_VERI = {}
for f in sorted(set(VADELI.values()) | {"ES=F", "NQ=F"}):
    try:
        VADELI_VERI[f], ort = vadeli_gunleri(f)
        durum = "✅ gece seansı geliyor" if ort >= 100 else "⚠️ gece verisi eksik görünüyor"
        print(f"{f:6s}: {len(VADELI_VERI[f])} gün, gece başına ort. {tr_sayi(ort, 0)} mum (tam gece ≈ 186) {durum}")
    except Exception as e:
        VADELI_VERI[f] = {}
        print(f"{f:6s}: alınamadı ({e})")
print("Not: Eylül ve Aralık'ta kontrat değişimi olur; aynı gün içindeki göreli ölçüler etkilenmez.")

In [ ]:
def bilanco_gunleri(sembol):
    """Bilançonun fiyata yansıdığı ilk seans: kapanış sonrası -> ertesi gün, açılış öncesi -> aynı gün."""
    try:
        e = yf.Ticker(sembol).get_earnings_dates(limit=12)
    except Exception:
        return set()
    if e is None or len(e) == 0:
        return set()
    gunler = set()
    for ts in e.index:
        ts = pd.Timestamp(ts)
        ts = ts.tz_convert("America/New_York") if ts.tzinfo else ts.tz_localize("America/New_York")
        d = pd.Timestamp(ts.date())
        if ts.hour >= 16:
            d = d + pd.offsets.BDay(1)
        gunler.add(d)
    return gunler

BILANCO = {s: bilanco_gunleri(s) for s in mag7}
for s in mag7:
    pencerede = [tr_tarih(d) for d in sorted(BILANCO[s])
                 if s in VERI and VERI[s] and VERI[s][0]["tarih"] <= d <= VERI[s][-1]["tarih"]]
    print(f"{s:5s}: pencerede bilanço günü: {', '.join(pencerede) if pencerede else 'yok'}")

## 3 · Simülasyon motoru
Bölüm 1'deki motorun aynısı, iki ekle: **VWAP iz süren çıkış** seçeneği ve her işlemin hem **kötümser** hem **iyimser** mum içi varsayımla hesaplanması (girişler aynı, yalnızca çıkışlar değişir). Kayma ETF'lerde ve hisselerde ayrı ayarlanır.

**VWAP iz süren çıkış:** girişten sonraki bir 5 dk mum VWAP'ın aleyhte tarafında kapanırsa bir sonraki mumun açılışında çıkılır. Asıl stop da yerinde durur.

In [ ]:
def gun_simule(g, or_dk, yon_kurali, stop_turu, atr_yuzde, son_giris_dk, komisyon, kayma_bps,
               ayni_mum="kötümser", cikis_turu="gün sonu", yon_zorla=None):
    k = or_dk // 5
    o, h, l, c, d_ = g["o"], g["h"], g["l"], g["c"], g["dk"]
    if len(o) <= k + 1:
        return None
    orh, orl, oro, orc, atr = h[:k].max(), l[:k].min(), o[0], c[k - 1], g["atr"]
    if yon_kurali.startswith("aralık"):
        if abs(orc - oro) < 1e-4 * oro:
            return None
        yon = 1 if orc > oro else -1
    else:
        yon = None
        for j in range(k, len(o)):
            yuk, asg = h[j] > orh, l[j] < orl
            if yuk and asg:
                return None
            if yuk or asg:
                yon = 1 if yuk else -1
                break
        if yon is None:
            return None
    if yon_zorla is not None:
        yon = yon_zorla
    seviye = orh if yon == 1 else orl

    giris = gj = None
    for j in range(k, len(o)):
        if d_[j] > son_giris_dk:
            break
        if (yon == 1 and h[j] >= seviye) or (yon == -1 and l[j] <= seviye):
            giris = max(seviye, o[j]) if yon == 1 else min(seviye, o[j])
            gj = j
            break
    if giris is None:
        return None

    if stop_turu.startswith("ATR"):
        stop = giris - yon * atr * atr_yuzde / 100
    elif stop_turu.startswith("aralığın karşı"):
        stop = orl if yon == 1 else orh
    else:
        stop = (orh + orl) / 2
    risk = (giris - stop) * yon
    if risk <= 0:
        return None

    cikis, neden, cj = c[-1], "gün sonu", len(o) - 1
    kotumser = (ayni_mum == "kötümser")
    vwap_cikis = cikis_turu.startswith("VWAP")
    for i in range(gj, len(o)):
        ilk = (i == gj)
        if not ilk and ((yon == 1 and o[i] <= stop) or (yon == -1 and o[i] >= stop)):
            cikis, neden, cj = o[i], "stop (boşluk)", i; break
        stop_degdi = (yon == 1 and l[i] <= stop) or (yon == -1 and h[i] >= stop)
        stop_kesin = stop_degdi and (not ilk or (yon == 1 and c[i] <= stop) or (yon == -1 and c[i] >= stop))
        if (kotumser and stop_degdi) or (not kotumser and stop_kesin):
            cikis, neden, cj = stop, "stop", i; break
        if vwap_cikis and not ilk:
            if (yon == 1 and c[i] < g["vwap"][i]) or (yon == -1 and c[i] > g["vwap"][i]):
                if i + 1 < len(o):
                    cikis, neden, cj = o[i + 1], "VWAP", i + 1
                else:
                    cikis, neden, cj = c[i], "VWAP", i
                break

    brut = (cikis - giris) * yon
    maliyet = 2 * komisyon + kayma_bps / 1e4 * (giris + cikis)
    return dict(tarih=g["tarih"], yon=yon, giris=giris, stop=stop, cikis=cikis, neden=neden,
                giris_saati=d_[gj], risk=risk, R_brut=brut / risk, R_net=(brut - maliyet) / risk,
                maliyet_R=maliyet / risk, or_genislik_atr=(orh - orl) / atr,
                goreli_hacim=g.get("goreli_hacim", np.nan))

def goreli_hacim_ekle(gunler, or_dk):
    k = or_dk // 5
    orv = [g["v"][:k].sum() for g in gunler]
    for i, g in enumerate(gunler):
        gecmis = orv[max(0, i - 14):i]
        g["goreli_hacim"] = orv[i] / np.mean(gecmis) if len(gecmis) == 14 and np.mean(gecmis) > 0 else np.nan

def calistir(or_dk, ayni_mum="kötümser", cikis_turu="gün sonu", yon_uretici=None):
    satirlar = []
    for s in VERI:
        goreli_hacim_ekle(VERI[s], or_dk)
        kayma = KAYMA_BPS_ETF if GRUP[s] == "ETF" else KAYMA_BPS_HISSE
        for g in VERI[s]:
            r = gun_simule(g, or_dk, YON_KURALI, STOP_TURU, ATR_YUZDE, dk(SON_GIRIS), KOMISYON_HISSE,
                           kayma, ayni_mum, cikis_turu, yon_uretici() if yon_uretici else None)
            if r:
                r["sembol"], r["grup"] = s, GRUP[s]
                satirlar.append(r)
    return pd.DataFrame(satirlar)

def islemler(or_dk, cikis_turu="gün sonu"):
    """Aynı girişler, iki mum içi varsayım: R_net_k (kötümser) ve R_net_i (iyimser)."""
    k = calistir(or_dk, "kötümser", cikis_turu)
    i = calistir(or_dk, "iyimser", cikis_turu)
    if k.empty:
        return k
    df = k.rename(columns={"R_net": "R_net_k", "R_brut": "R_brut_k", "neden": "neden_k"})
    df = df.merge(i[["sembol", "tarih", "R_net", "R_brut"]].rename(
        columns={"R_net": "R_net_i", "R_brut": "R_brut_i"}), on=["sembol", "tarih"])
    return df

def ozet(df, kol="R_net_k"):
    r = df[kol]
    if len(r) == 0:
        return {}
    kaz, kay = r[r > 0].sum(), -r[r < 0].sum()
    return {"işlem": len(r), "isabet": (r > 0).mean(), "ort. R net (kötümser)": df["R_net_k"].mean(),
            "ort. R net (iyimser)": df["R_net_i"].mean(), "kâr faktörü (kötümser)": kaz / kay if kay > 0 else np.nan,
            "t-ist. (kötümser)": r.mean() / (r.std(ddof=1) / np.sqrt(len(r))) if len(r) > 2 else np.nan}

## 4 · Gün bağlamı özellikleri
Her işleme şu sütunlar eklenir (vadeli = sembolün eşleştiği vadeli):
- `vix_onceki`: önceki günün VIX kapanışı.
- `acik_hava`: girişten önce tamamlanmış mumlarda vadeli, işlem yönünde gece yükseğini (uzun) / düşüğünü (kısa) aşmış mı? (Giriş mumu dahil edilmez: o mumun içinde aşma girişten sonra da olmuş olabilir.)
- `acilis_gece_uyum`: vadelinin 09:30 fiyatı gece aralığının işlem yönündeki yarısında mı?
- `gece_getiri_uyum`: vadelinin önceki seans kapanışından 09:30'a getirisi işlem yönünde mi?
- `es_nq_uyum`: ES ve NQ'nun açılış aralığı mumu (aynı süre) ikisi de işlem yönünde mi?
- `bilanco_gunu`: Mag7 için bilançonun fiyata yansıdığı ilk seans mı?

In [ ]:
def vadeli_or_yonu(fut, tarih, or_dk):
    d = VADELI_VERI.get(fut, {}).get(tarih)
    if d is None or len(d["o"]) < or_dk // 5 or d["dk"][0] != ACILIS_DK:
        return np.nan
    return np.sign(d["c"][or_dk // 5 - 1] - d["o"][0])

def ozellik_ekle(df, or_dk):
    df = df.copy()
    df["vix_onceki"] = [VIX_ONCEKI.get(t, np.nan) for t in df["tarih"]]
    ah, au, gu, en, bg = [], [], [], [], []
    for _, r in df.iterrows():
        d = VADELI_VERI.get(VADELI[r["sembol"]], {}).get(r["tarih"])
        if d is None:
            ah.append(np.nan); au.append(np.nan); gu.append(np.nan)
        else:
            maske = d["dk"] < r["giris_saati"]   # yalnızca girişten ÖNCE tamamlanmış mumlar
            if r["yon"] == 1:
                ah.append(float(d["h"][maske].max() >= d["onh"]) if maske.any() else np.nan)
            else:
                ah.append(float(d["l"][maske].min() <= d["onl"]) if maske.any() else np.nan)
            orta = (d["onh"] + d["onl"]) / 2
            au.append(float((d["p0930"] > orta) == (r["yon"] == 1)))
            gu.append(float(np.sign(d["p0930"] - d["onceki_kapanis"]) == r["yon"]))
        es, nq = vadeli_or_yonu("ES=F", r["tarih"], or_dk), vadeli_or_yonu("NQ=F", r["tarih"], or_dk)
        en.append(np.nan if np.isnan(es) or np.isnan(nq) else float(es == r["yon"] and nq == r["yon"]))
        bg.append(float(r["tarih"] in BILANCO.get(r["sembol"], set())) if r["grup"] == "Mag7" else np.nan)
    df["acik_hava"], df["acilis_gece_uyum"], df["gece_getiri_uyum"] = ah, au, gu
    df["es_nq_uyum"], df["bilanco_gunu"] = en, bg
    return df

ISLEM = {}
for or_dk in [5, 15]:
    ISLEM[or_dk] = ozellik_ekle(islemler(or_dk), or_dk)
    d = ISLEM[or_dk]
    dolu = {k: tr_yuzde(d[k].notna().mean()) for k in ["vix_onceki", "acik_hava", "es_nq_uyum"]}
    print(f"{or_dk} dk: {len(d)} işlem · özellik doluluğu: {dolu}")

## 5 · 5 dk mı 15 dk mı? (hisse hisse)
Her sembol için iki aralık yan yana. **Okuma kuralı:**
- Kötümser ve iyimser aynı işareti veriyorsa 5 dk veri o sembol için bir şey söylüyor demektir; farklıysa söylemiyor.
- **p (rastgele):** aynı günlerde yazı-tura yönüyle yapılan işlemlerin kaçta kaçı stratejiye eşit ya da daha iyi. Küçükse ilk aralığın yönü o sembolde bilgi taşıyor.
- Sembol başına ~40–55 işlem var; tek sembol sonuçlarını kesin değil, **ipucu** olarak oku.

In [ ]:
def rastgele_p(or_dk, gercek):
    """Sembol, grup ve tüm evren için rastgele yönlü kıyas p-değerleri (kötümser varsayım)."""
    kayit = []
    for _ in range(RASTGELE_TEKRAR):
        df = calistir(or_dk, "kötümser", yon_uretici=lambda: int(rng.choice([-1, 1])))
        sat = df.groupby("sembol")["R_net"].mean().to_dict()
        sat.update(df.groupby("grup")["R_net"].mean().to_dict())
        sat["TÜMÜ"] = df["R_net"].mean()
        kayit.append(sat)
    kayit = pd.DataFrame(kayit)
    p, med = {}, {}
    for anahtar in kayit.columns:
        g = gercek.get(anahtar, np.nan)
        x = kayit[anahtar].dropna().values
        p[anahtar] = (np.sum(x >= g) + 1) / (len(x) + 1) if len(x) else np.nan
        med[anahtar] = np.median(x) if len(x) else np.nan
    return p, med

TABLO5 = []
for or_dk in [5, 15]:
    d = ISLEM[or_dk]
    gercek = d.groupby("sembol")["R_net_k"].mean().to_dict()
    gercek.update(d.groupby("grup")["R_net_k"].mean().to_dict())
    gercek["TÜMÜ"] = d["R_net_k"].mean()
    p, med = rastgele_p(or_dk, gercek)
    for anahtar in semboller + ["ETF", "Mag7", "TÜMÜ"]:
        alt = d if anahtar == "TÜMÜ" else d[(d["sembol"] == anahtar) | (d["grup"] == anahtar)]
        if alt.empty:
            continue
        o_ = ozet(alt)
        TABLO5.append({"sembol": anahtar, "OR": f"{or_dk} dk", "işlem": o_["işlem"], "isabet": o_["isabet"],
                       "net R kötümser": o_["ort. R net (kötümser)"], "net R iyimser": o_["ort. R net (iyimser)"],
                       "yön katkısı": o_["ort. R net (kötümser)"] - med.get(anahtar, np.nan),
                       "p (rastgele)": p.get(anahtar, np.nan)})
TABLO5 = pd.DataFrame(TABLO5)

def isaret(r):
    a, b = r["net R kötümser"], r["net R iyimser"]
    return "+" if a > 0 and b > 0 else ("−" if a < 0 and b < 0 else "belirsiz")
TABLO5["karar"] = TABLO5.apply(isaret, axis=1)

for or_ in ["5 dk", "15 dk"]:
    display(Markdown(f"**{or_} ORB · stop: {STOP_TURU}**"))
    display(tr_tablo(TABLO5[TABLO5["OR"] == or_].drop(columns="OR").set_index("sembol")))

kars = TABLO5.pivot(index="sembol", columns="OR", values="net R kötümser").reindex(semboller + ["ETF", "Mag7", "TÜMÜ"])
kars["fark (5 − 15)"] = kars["5 dk"] - kars["15 dk"]
display(Markdown("**Net R (kötümser): 5 dk ile 15 dk karşılaştırması**"))
display(tr_tablo(kars))

## 6 · Hangi teyit işe yarıyor?
Her filtre işlemleri ikiye böler: **geçti** ve **kaldı**. Filtre işe yarıyorsa "geçti" grubunun ortalaması belirgin şekilde yüksek olmalı.

**Permütasyon testi:** filtre etiketleri karıştırılır ve "geçti − kaldı" farkı binlerce kez yeniden hesaplanır. Gerçek farkın bu karışık farklardan ne kadar sıradışı olduğu p-değerini verir. Karıştırma **gün bazında** yapılır: aynı gündeki 10 işlem ortak piyasa hareketini ve ortak vadeli etiketini paylaşır, bağımsız değildir. İşlem bazında karıştırmak p-değerlerini olduğundan küçük gösterir (sahte veri testinde bu yüzden yanlış alarmlar çıktı).

**Çoklu test uyarısı:** 6 filtre × 2 aralık = 12 test. Hiçbir filtre işe yaramasa bile p < 0,10 eşiğinde ortalama 1 tanesi şans eseri "anlamlı" çıkar. Bu yüzden p-değerleri **Benjamini–Hochberg** yöntemiyle düzeltilir (q sütunu): q < 0,10, "aday" listesindekilerin en fazla %10'unun şans eseri olmasını hedefler. Bir filtreyi **aday** saymak için dört koşul aranır:
1. q < 0,10 (tüm testler birlikte düşünülerek),
2. fark iki mum içi varsayımda da pozitif,
3. "geçti" grubunda en az 20 işlem,
4. fark örneklemin **ilk ve ikinci yarısında ayrı ayrı** pozitif (dönemden bağımsız mı?).

Not defteri, filtrelerin hiçbir gerçek etkisinin olmadığı sahte veriyle 6 kez çalıştırıldı. Düzeltmesiz p < 0,10 kuralı 6 çalıştırmanın 3'ünde en az bir sahte "keşif" üretti. q düzeltmesi ve gün bazında permütasyonla bu oran 8 çalıştırmada 1'e indi (hedeflenen düzey ~%10). Yine de "aday" kural değil, daha uzun veride doğrulanacak bir hipotezdir.

In [ ]:
FILTRELER = {
    f"Göreli hacim ≥ {tr_sayi(GH_ESIK, 1)}": lambda d: (d["goreli_hacim"] >= GH_ESIK).where(d["goreli_hacim"].notna()),
    "VIX medyan üstü": lambda d: (d["vix_onceki"] > d["vix_onceki"].median()).where(d["vix_onceki"].notna()),
    "Gece aralığı dışına kırılım": lambda d: d["acik_hava"],
    "Açılış gece ortasının işlem yönünde": lambda d: d["acilis_gece_uyum"],
    "Gece getirisi işlem yönünde": lambda d: d["gece_getiri_uyum"],
    "ES ve NQ aralığı işlem yönünde": lambda d: d["es_nq_uyum"],
}

def permutasyon_p(r, m, gun, tekrar):
    """Gün bazında permütasyon: aynı günün işlemleri birlikte hareket eder (ortak piyasa ve ortak
    vadeli etiketi). Her tekrarda günlerin R dizileri, başka bir günün etiket dizisiyle eşleştirilir;
    gün içi sıra korunur, gün uzunluğu farklıysa etiketler döngüsel olarak taşınır."""
    gozlenen = r[m].mean() - r[~m].mean()
    gunler = pd.unique(gun)
    r_g = [r[gun == g] for g in gunler]
    m_g = [m[gun == g] for g in gunler]
    say = 0
    for _ in range(tekrar):
        sira = rng.permutation(len(gunler))
        mm = np.concatenate([np.resize(rng.permutation(m_g[j]), len(r_g[i])) for i, j in enumerate(sira)])
        rr = np.concatenate(r_g)
        if mm.all() or (~mm).all():
            continue
        if rr[mm].mean() - rr[~mm].mean() >= gozlenen:
            say += 1
    return gozlenen, (say + 1) / (tekrar + 1)

FILTRE_SONUC = []
for or_dk in [5, 15]:
    d = ISLEM[or_dk]
    for ad, fn in FILTRELER.items():
        m = fn(d)
        gecerli = m.notna()
        if gecerli.sum() < 10:
            continue
        dd, mm = d[gecerli], m[gecerli].astype(bool).values
        if mm.sum() < 5 or (~mm).sum() < 5:
            continue
        fark_k, p = permutasyon_p(dd["R_net_k"].values, mm, dd["tarih"].values, PERMUTASYON_TEKRAR)
        fark_i = dd["R_net_i"].values[mm].mean() - dd["R_net_i"].values[~mm].mean()
        ortanca_tarih = dd["tarih"].sort_values().iloc[len(dd) // 2]
        yari = []
        for parca in [dd["tarih"] < ortanca_tarih, dd["tarih"] >= ortanca_tarih]:
            r_, m_ = dd.loc[parca, "R_net_k"].values, mm[parca.values]
            yari.append(r_[m_].mean() - r_[~m_].mean() if m_.any() and (~m_).any() else np.nan)
        FILTRE_SONUC.append({"OR": f"{or_dk} dk", "filtre": ad, "geçti n": int(mm.sum()), "kaldı n": int((~mm).sum()),
                             "geçti R (kötümser)": dd["R_net_k"].values[mm].mean(),
                             "kaldı R (kötümser)": dd["R_net_k"].values[~mm].mean(),
                             "fark (kötümser)": fark_k, "fark (iyimser)": fark_i,
                             "ilk yarı fark": yari[0], "ikinci yarı fark": yari[1], "p": p})
FILTRE_SONUC = pd.DataFrame(FILTRE_SONUC)

def bh_q(p):
    """Benjamini–Hochberg q-değerleri: çoklu testte yanlış keşif oranını kontrol eder."""
    p = np.asarray(p, float); n = len(p); sira = np.argsort(p)
    q = np.empty(n); onceki = 1.0
    for rank in range(n, 0, -1):
        i = sira[rank - 1]
        onceki = min(onceki, p[i] * n / rank)
        q[i] = onceki
    return q

FILTRE_SONUC["q"] = bh_q(FILTRE_SONUC["p"].values)
FILTRE_SONUC["aday"] = np.where((FILTRE_SONUC["q"] < 0.10) & (FILTRE_SONUC["fark (kötümser)"] > 0)
                                & (FILTRE_SONUC["fark (iyimser)"] > 0) & (FILTRE_SONUC["geçti n"] >= 20)
                                & (FILTRE_SONUC["ilk yarı fark"] > 0) & (FILTRE_SONUC["ikinci yarı fark"] > 0), "✅ aday", "")
for or_ in ["5 dk", "15 dk"]:
    display(Markdown(f"**{or_} ORB · filtreler (tüm evren)**"))
    display(tr_tablo(FILTRE_SONUC[FILTRE_SONUC["OR"] == or_].drop(columns="OR").set_index("filtre")))

In [ ]:
#@title 🔎 Filtreler grup bazında (ETF / Mag7) — kötümser varsayım
satir = []
for or_dk in [5, 15]:
    d = ISLEM[or_dk]
    for grup in ["ETF", "Mag7"]:
        dg = d[d["grup"] == grup]
        for ad, fn in FILTRELER.items():
            m = fn(dg)
            g_ = m.notna()
            if g_.sum() < 10:
                continue
            mm = m[g_].astype(bool)
            r = dg.loc[g_, "R_net_k"]
            satir.append({"OR": f"{or_dk} dk", "grup": grup, "filtre": ad, "geçti n": int(mm.sum()),
                          "geçti R": r[mm].mean() if mm.any() else np.nan,
                          "kaldı R": r[~mm].mean() if (~mm).any() else np.nan})
GT = pd.DataFrame(satir)
GT["fark"] = GT["geçti R"] - GT["kaldı R"]
pv = GT.pivot_table(index="filtre", columns=["OR", "grup"], values="fark")
pv.columns = [f"{a} · {b}" for a, b in pv.columns]
display(Markdown("**Geçti − kaldı farkı (ort. net R, kötümser)**"))
display(tr_tablo(pv))

## 7 · VWAP: giriş filtresi değil, çıkış kuralı
Aynı girişler iki çıkışla karşılaştırılır: **gün sonu** (makale) ve **VWAP iz süren**. VWAP çıkışı kayıpları kısaltır ama büyük trend günlerinde kârı da erken keser; ORB'nin kârı seyrek büyük günlerden geldiği için bu takas belirleyicidir.

In [ ]:
VW = []
for or_dk in [5, 15]:
    a = ISLEM[or_dk]
    b = islemler(or_dk, "VWAP iz süren")
    for ad, d in [("gün sonu", a), ("VWAP iz süren", b)]:
        for grup in ["ETF", "Mag7", "TÜMÜ"]:
            alt = d if grup == "TÜMÜ" else d[d["grup"] == grup]
            o_ = ozet(alt)
            VW.append({"OR": f"{or_dk} dk", "çıkış": ad, "grup": grup, "işlem": o_["işlem"], "isabet": o_["isabet"],
                       "net R kötümser": o_["ort. R net (kötümser)"], "net R iyimser": o_["ort. R net (iyimser)"],
                       "kâr faktörü": o_["kâr faktörü (kötümser)"]})
VW = pd.DataFrame(VW)
for or_ in ["5 dk", "15 dk"]:
    display(Markdown(f"**{or_} ORB · çıkış karşılaştırması**"))
    display(tr_tablo(VW[VW["OR"] == or_].drop(columns="OR").set_index(["grup", "çıkış"])))

## 8 · Mag7 bilanço günleri
Bilanço günleri Stocks in Play mantığının özüdür (haber + hacim), ama aynı zamanda en sert ve en manipülatif açılışlardır. Pencerede sembol başına en fazla 1 bilanço olduğu için bu tablo yalnızca **gözlem**dir.

In [ ]:
satir = []
for or_dk in [5, 15]:
    d = ISLEM[or_dk]
    dm = d[d["grup"] == "Mag7"]
    for etiket, alt in [("bilanço günü", dm[dm["bilanco_gunu"] == 1]), ("diğer günler", dm[dm["bilanco_gunu"] == 0])]:
        if alt.empty:
            continue
        o_ = ozet(alt)
        satir.append({"OR": f"{or_dk} dk", "gün": etiket, "işlem": o_["işlem"], "isabet": o_["isabet"],
                      "net R kötümser": o_["ort. R net (kötümser)"], "net R iyimser": o_["ort. R net (iyimser)"]})
if satir:
    display(tr_tablo(pd.DataFrame(satir).set_index(["OR", "gün"])))
    bl = ISLEM[15][(ISLEM[15]["bilanco_gunu"] == 1)][["sembol", "tarih", "yon", "R_net_k", "R_net_i"]]
    if not bl.empty:
        bl = bl.assign(tarih=bl["tarih"].map(tr_tarih), yon=bl["yon"].map({1: "uzun", -1: "kısa"}))
        display(Markdown("**15 dk · bilanço günü işlemleri**"))
        display(tr_tablo(bl.set_index(["sembol", "tarih"])))
else:
    print("Pencerede bilanço günü yok ya da tarihler alınamadı.")

## 9 · Özet ve X gönderisi
Özet otomatik üretilir; sonuçlar her çalıştırmada değişebilir.

In [ ]:
def ozet_cumleleri():
    c = []
    t = TABLO5.set_index(["sembol", "OR"])
    for grup in ["ETF", "Mag7", "TÜMÜ"]:
        v5, v15 = t.loc[(grup, "5 dk")], t.loc[(grup, "15 dk")]
        c.append(f"{grup}: 5 dk {tr_sayi(v5['net R kötümser'], 2)} R ({v5['karar']}), "
                 f"15 dk {tr_sayi(v15['net R kötümser'], 2)} R ({v15['karar']}), "
                 f"p(5 dk)={tr_sayi(v5['p (rastgele)'], 2)}, p(15 dk)={tr_sayi(v15['p (rastgele)'], 2)}")
    adaylar = FILTRE_SONUC[FILTRE_SONUC["aday"] != ""]
    if adaylar.empty:
        c.append("Hiçbir filtre aday koşullarını (q<0,10, iki varsayımda ve iki yarıda pozitif, ≥20 işlem) sağlamadı.")
    else:
        for _, r in adaylar.iterrows():
            c.append(f"Aday filtre: {r['filtre']} ({r['OR']}): fark {tr_sayi(r['fark (kötümser)'], 2)} R, "
                     f"p={tr_sayi(r['p'], 3)}, q={tr_sayi(r['q'], 3)}, {r['geçti n']} işlem")
    tum = VW[VW["grup"] == "TÜMÜ"].set_index(["OR", "çıkış"])
    for or_ in ["5 dk", "15 dk"]:
        c.append(f"VWAP çıkışı {or_}: {tr_sayi(tum.loc[(or_, 'VWAP iz süren'), 'net R kötümser'], 2)} R "
                 f"(gün sonu {tr_sayi(tum.loc[(or_, 'gün sonu'), 'net R kötümser'], 2)} R)")
    return c

OZET = ozet_cumleleri()
display(Markdown("**Özet**\n\n" + "\n".join(f"- {x}" for x in OZET)))

d15 = ISLEM[15]
bas, son = min(ISLEM[5]["tarih"].min(), d15["tarih"].min()), max(ISLEM[5]["tarih"].max(), d15["tarih"].max())
t = TABLO5.set_index(["sembol", "OR"])
adaylar = FILTRE_SONUC[FILTRE_SONUC["aday"] != ""]
gonderi = (
    "Açılış aralığı kırılımı (ORB): 5 dk mı 15 dk mı?\n"
    "$SPY $QQQ $IWM + Mag7 · " + f"{tr_tarih(bas)}–{tr_tarih(son)}\n\n"
    f"ETF'ler: 5 dk {tr_sayi(t.loc[('ETF', '5 dk'), 'net R kötümser'], 2)} R · 15 dk {tr_sayi(t.loc[('ETF', '15 dk'), 'net R kötümser'], 2)} R\n"
    f"Mag7: 5 dk {tr_sayi(t.loc[('Mag7', '5 dk'), 'net R kötümser'], 2)} R · 15 dk {tr_sayi(t.loc[('Mag7', '15 dk'), 'net R kötümser'], 2)} R\n"
    "(işlem başına, maliyet sonrası)\n\n"
    + ("Öne çıkan teyit: " + ", ".join(sorted(set(adaylar["filtre"]))) if not adaylar.empty
       else "Test ettiğim 6 teyidin hiçbiri şanstan ayırt edilemedi.")
    + "\nVWAP girişte işe yaramaz: 15 dk sonunda zaten aralığın içinde.\n\n"
    "Öğrenme amaçlıdır, yatırım tavsiyesi değildir."
)
print("KOPYALAMAK İÇİN DÜZ METİN\n" + "-" * 40)
print(gonderi)
print("-" * 40, f"\n{len(gonderi)} karakter")

## 10 · Sınırlar ve sonraki adımlar
- **Veri kısa:** ~60 gün tek bir rejimdir; filtre testleri az işlemle yapılır. Aday çıkan bir filtre, daha uzun veride doğrulanana kadar **kural değil hipotezdir**.
- **5 dk aralık** tek mumdur; 5 dk sonuçlarında mum içi belirsizlik 15 dk'dan büyüktür.
- **Vadeli verisi** yfinance sürekli kontratıdır; gece verisi kontrolü 2. bölümde yazdırılır. Eksikse vadeli filtreleri boş kalır.
- **Bilanço tarihleri** yfinance'ten gelir ve eksik olabilir.

**Sıradaki:** aday çıkan filtreleri daha uzun ve 1 dakikalık veriyle doğrulamak (ücretsiz/ucuz veri kaynağı araştırması); ardından Noise Area (Bölüm 3).

> Öğrenme amaçlıdır, yatırım tavsiyesi değildir.